> VIGOR 2.12.1 example. Install with `pip install "VIGOR[io,progress,plot] @ git+https://github.com/sanchestm/VIGOR"` (see the [README](../../../README.md)). Outputs are cleared: this notebook was checked against the 2.12.1 API but not re-executed for this release.

# HS project 00 — setup, phenotypes and canonical xarray handles

In [ ]:
from pathlib import Path
import sys, os, json, warnings
import numpy as np
import pandas as pd
import xarray as xr

# Find ROOT/vigor_run/project_config.py regardless of notebook cwd.
def _find_run(start=None):
    p=Path(os.environ.get('VIGOR_PROJECT_ROOT', os.environ.get('DOMINO_PROJECT_ROOT', Path.cwd()))) if start is None else Path(start)
    for q in [p,*p.parents]:
        cand=q/'vigor_run'/'project_config.py'
        if cand.exists(): return cand.parent
        if q.name=='vigor_run' and (q/'project_config.py').exists(): return q
    raise FileNotFoundError('Set VIGOR_PROJECT_ROOT or place vigor_run under the project root.')
RUN=_find_run(); sys.path.insert(0,str(RUN))
from project_config import *
import vigor
print('Project root:',ROOT)
print('VIGOR:',vigor.__version__)
assert tuple(map(int, vigor.__version__.split('.')[:3])) >= (2, 12, 1), vigor.__version__

In [ ]:
TRAITS = [
    'oxy_add_index_pc1','oxy_lga_iti_median_11_14','oxy_lga_mean_11_14','oxy_lga_pc1',
    'oxy_lga_total_intake','oxy_total_intake','clg_pc1_ai','clg_lga_total_intake',
    'clg_total_intake','clg_lga_iti_median_11_14','clg_lga_mean_11_14','clg_pc1_lga',
    'dd_auc','dd_exponential_k','dd_hyperbolic_k','pf_dd_idptauc','pf_dd_log_k',
    'her_total_heroin_consumption','her_escalation_of_heroin_intake_12h',
    'loc_p50_shelly_flagel_2014_d1_first_15','cit_inta_esc','cit_inta_esc_olivier_avg',
    'cit_inta_inf_tot','cit_inta_index_esc','cit_life_tot_rewards','cit_inta_inf_d15'
]

In [ ]:
def _read_traits(path):
    x=pd.read_csv(path,index_col=0);x.index=x.index.astype(str);x.columns=x.columns.str.replace('regressedlr_','',regex=False)
    return x.astype(float)

if all(p.exists() for p in (TRAIN_CSV,TEST_ROWS_CSV,TEST_ELEMS_CSV)):
    train=_read_traits(TRAIN_CSV).reindex(columns=TRAITS)
    test_rows=_read_traits(TEST_ROWS_CSV).reindex(index=train.index,columns=TRAITS)
    test_elems=_read_traits(TEST_ELEMS_CSV).reindex(index=train.index,columns=TRAITS)
    full=train.combine_first(test_rows).combine_first(test_elems)
elif PROCESSED_DATA_CSV.exists():
    from vigor import causal
    pdr=pd.read_csv(PROCESSED_DATA_CSV,dtype={'rfid':str}).set_index('rfid')
    cols=['regressedlr_'+t for t in TRAITS]
    raw=pdr[cols].rename(columns=lambda c:c.replace('regressedlr_',''))
    mr,me,mc=causal.create_dual_masking_scheme(raw,row_pct=ROW_PCT,elem_pct=ELEM_PCT,seed=SEED)
    train=raw.mask(mc);test_rows=raw.where(mr);test_elems=raw.where(me);full=raw
    train.to_csv(TRAIN_CSV);train.to_csv(TRAIN_RGBLUP_CSV,index_label='ID');test_rows.to_csv(TEST_ROWS_CSV);test_elems.to_csv(TEST_ELEMS_CSV)
else:
    raise FileNotFoundError('Need processed_data_ready.csv or existing train/test CSVs')

pheno=xr.Dataset({'full':(('sample','trait'),full.to_numpy()),'train':(('sample','trait'),train.to_numpy()),'test_rows':(('sample','trait'),test_rows.to_numpy()),'test_elems':(('sample','trait'),test_elems.to_numpy())},coords={'sample':train.index.astype(str),'trait':train.columns.astype(str)})
pheno

# Persist the exact project masks once. All later notebooks can recover the same
# sample/trait coordinates without reparsing or regenerating the split.
ANALYSIS_STORE.parent.mkdir(parents=True,exist_ok=True)
store=vigor.open_store(ANALYSIS_STORE,mode='a',lazy=False)
store.write_dataset(pheno,group='phenotype/project_masks',mode='a')
print('Persisted phenotype masks:',ANALYSIS_STORE,':: phenotype/project_masks')


In [ ]:
# BED remains packed on disk; this is a lazy labeled xarray view, not a genotype copy.
geno=vigor.open_genotype_dataset(PLINK_PREFIX,source_format='plink1',variant_chunk=16_384)
print(geno)
print('GT backing type:',type(geno.GT.data))
print('samples/variants:',geno.sizes['sample'],geno.sizes['variant'])


The BED file remains the packed physical backend. Nothing in this notebook requires a full PLINK→Zarr genotype conversion. Derived matrices and probability-rich layers can be written to Zarr independently.